# Phase 2 Stress-Regime Candidate Analysis

This notebook supports Milestone 7. It does **not** assume that a volatility spike is automatically a crisis. Instead, it explores which observable market-stress signals tend to coincide with future high-volatility / crisis-like episodes.

Purpose:

1. characterize high-volatility and crisis-like candidate episodes;
2. rank available stress indicators such as VIX, drawdown, selloff pressure, and volume stress;
3. test whether a transparent four-level warning ladder is defensible;
4. generate tables and plots for the final application narrative.

Guardrail: this is an interpretation/application layer. Track A metrics still evaluate the volatility forecast substrate.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.splits import chronological_tabular_split

out_table_dir = Path('results/tables')
out_fig_dir = Path('results/figures')
out_table_dir.mkdir(parents=True, exist_ok=True)
out_fig_dir.mkdir(parents=True, exist_ok=True)

## 1. Load data and define stress-oriented observables

We keep labels rule-based and train-calibrated. No hand-labeled crisis taxonomy is used.

In [ ]:
target = 'future_rv_20d'
df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

def make_stress_features(frame):
    out = frame.copy().reset_index(drop=True)
    out['date'] = pd.to_datetime(out['date'])
    out['spy_selloff_stress'] = -out['spy_log_return']
    out['spy_oc_selloff_stress'] = -out['spy_log_oc_return']
    out['drawdown_stress'] = -out['spy_drawdown_20d'] if out['spy_drawdown_20d'].median() < 0 else out['spy_drawdown_20d']
    out['volume_stress'] = out['spy_log_volume_change'].abs()
    out['vix_level_stress'] = out['vix_close']
    out['vix_change_stress'] = out['vix_log_change'].clip(lower=0.0)
    out['vix_range_stress'] = out['vix_log_hl_range']
    out['realized_vol_stress'] = out['rv_20d']
    return out

splits = {k: make_stress_features(v) for k, v in splits.items()}
train, val, test = splits['train'], splits['val'], splits['test']
print(train.shape, val.shape, test.shape)
train[['date', target]].head()

## 2. Define outcome events from future realized volatility

These are not subjective crisis labels. They are operational outcome events. The crisis-like candidate threshold is deliberately conservative.

In [ ]:
thresholds = {
    'high_vol_q80': float(train[target].quantile(0.80)),
    'extreme_vol_q90': float(train[target].quantile(0.90)),
    'crisis_candidate_q95': float(train[target].quantile(0.95)),
}
thresholds

for split in [train, val, test]:
    split['future_high_vol_event_q80'] = split[target] > thresholds['high_vol_q80']
    split['future_extreme_vol_event_q90'] = split[target] > thresholds['extreme_vol_q90']
    split['future_crisis_candidate_q95'] = split[target] > thresholds['crisis_candidate_q95']

event_rates = pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'q80_event_rate': [train['future_high_vol_event_q80'].mean(), val['future_high_vol_event_q80'].mean(), test['future_high_vol_event_q80'].mean()],
    'q90_event_rate': [train['future_extreme_vol_event_q90'].mean(), val['future_extreme_vol_event_q90'].mean(), test['future_extreme_vol_event_q90'].mean()],
    'q95_event_rate': [train['future_crisis_candidate_q95'].mean(), val['future_crisis_candidate_q95'].mean(), test['future_crisis_candidate_q95'].mean()],
})
event_rates

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train[target], bins=80, alpha=0.6, label='train')
for name, thr in thresholds.items():
    ax.axvline(thr, linestyle='--', label=name)
ax.set_title('Train-calibrated future volatility thresholds')
ax.set_xlabel(target)
ax.set_ylabel('count')
ax.legend()
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_future_vol_thresholds_hist.png', dpi=180)
plt.show()

## 3. Rank candidate stress indicators

For each signal, we test whether train-q80/q90 threshold crossings identify future high-volatility or crisis-like events. This is feature screening, not an optimized classifier.

In [ ]:
candidate_features = [
    'rv_5d', 'rv_10d', 'rv_20d', 'rv_60d', 'rv_ratio_5_20', 'rv_ratio_20_60',
    'rv_slope_5_20', 'rv_slope_20_60', 'spy_abs_log_return', 'spy_squared_log_return',
    'spy_log_hl_range', 'spy_selloff_stress', 'spy_oc_selloff_stress', 'drawdown_stress',
    'spy_log_volume', 'spy_log_volume_change', 'volume_stress',
    'vix_close', 'vix_log_change', 'vix_abs_log_change', 'vix_log_hl_range',
    'vix_ma_5d', 'vix_std_5d', 'vix_ma_20d', 'vix_std_20d',
    'vix_level_stress', 'vix_change_stress', 'vix_range_stress', 'realized_vol_stress',
]
candidate_features = [c for c in candidate_features if c in train.columns]

def flag_metrics(signal_flag, event_flag):
    signal = np.asarray(signal_flag, dtype=bool)
    event = np.asarray(event_flag, dtype=bool)
    tp = int(np.sum(signal & event))
    fp = int(np.sum(signal & ~event))
    fn = int(np.sum(~signal & event))
    tn = int(np.sum(~signal & ~event))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1, signal_rate=float(signal.mean()))

rows = []
events = ['future_high_vol_event_q80', 'future_extreme_vol_event_q90', 'future_crisis_candidate_q95']
for feature in candidate_features:
    q80 = float(train[feature].quantile(0.80))
    q90 = float(train[feature].quantile(0.90))
    for event in events:
        for qname, thr in [('q80', q80), ('q90', q90)]:
            m = flag_metrics(test[feature] > thr, test[event])
            corr = float(np.corrcoef(test[feature], test[target])[0, 1]) if test[feature].std() > 0 else np.nan
            rows.append({
                'feature': feature, 'event': event, 'feature_threshold': qname, 'threshold_value': thr,
                'corr_with_future_rv_20d': corr, **m,
            })
ranking = pd.DataFrame(rows)
ranking.to_csv(out_table_dir / 'phase2_stress_feature_ranking_notebook.csv', index=False)
ranking.sort_values(['event', 'f1'], ascending=[True, False]).head(15)

In [ ]:
plot_rank = (ranking[ranking['event'] == 'future_extreme_vol_event_q90']
             .sort_values('f1', ascending=False)
             .head(12)
             .copy())
plot_rank['label'] = plot_rank['feature'] + ' / ' + plot_rank['feature_threshold']

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_rank['label'][::-1], plot_rank['f1'][::-1])
ax.set_title('Feature-screening F1 for future extreme-volatility events')
ax.set_xlabel('F1')
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_stress_feature_ranking_f1.png', dpi=180)
plt.show()

## 4. Profile top future-volatility episodes

This is the sanity check: do top future-volatility episodes look like known stress conditions across VIX, drawdown, selloff, volume, and realized-volatility state?

In [ ]:
profile_cols = [
    'date', target, 'rv_20d', 'vix_close', 'vix_ma_20d', 'vix_std_20d',
    'drawdown_stress', 'spy_selloff_stress', 'spy_log_hl_range', 'volume_stress',
    'future_high_vol_event_q80', 'future_extreme_vol_event_q90', 'future_crisis_candidate_q95',
]
top_future = test.sort_values(target, ascending=False).head(30)[profile_cols]
top_future.to_csv(out_table_dir / 'phase2_top_future_volatility_profile_notebook.csv', index=False)
top_future.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'], test[target], label='future_rv_20d', linewidth=1.3)
ax.axhline(thresholds['high_vol_q80'], linestyle='--', label='q80')
ax.axhline(thresholds['extreme_vol_q90'], linestyle='--', label='q90')
ax.axhline(thresholds['crisis_candidate_q95'], linestyle='--', label='q95')
crisis_like = test[test['future_crisis_candidate_q95']]
ax.scatter(crisis_like['date'], crisis_like[target], marker='o', s=35, label='q95 candidate')
ax.set_title('Future realized volatility and candidate crisis-like episodes')
ax.set_ylabel(target)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_future_volatility_candidate_episodes.png', dpi=180)
plt.show()

## 5. Build transparent stress-score ladder

This ladder is operational: Normal → Watch → Warning → Crisis-like. It is not a claim that the market has exactly four latent regimes.

In [ ]:
def q(col, p):
    return float(train[col].quantile(p))

timeline = test.copy()
timeline['flag_realized_vol'] = timeline['rv_20d'] > q('rv_20d', 0.80)
timeline['flag_vix_level'] = timeline['vix_close'] > q('vix_close', 0.80)
timeline['flag_vix_range'] = timeline['vix_log_hl_range'] > q('vix_log_hl_range', 0.80)
timeline['flag_drawdown'] = timeline['drawdown_stress'] > q('drawdown_stress', 0.80)
timeline['flag_selloff'] = timeline['spy_selloff_stress'] > q('spy_selloff_stress', 0.80)
timeline['flag_volume'] = timeline['volume_stress'] > q('volume_stress', 0.80)
timeline['flag_vol_acceleration'] = timeline['rv_slope_5_20'] > q('rv_slope_5_20', 0.80)

flag_cols = [c for c in timeline.columns if c.startswith('flag_')]
timeline['stress_score'] = timeline[flag_cols].sum(axis=1)
timeline['stress_ladder'] = pd.cut(
    timeline['stress_score'],
    bins=[-0.1, 0.5, 1.5, 2.5, len(flag_cols) + 0.5],
    labels=['normal', 'watch', 'warning', 'crisis_like'],
).astype(str)
timeline['direction_tag'] = 'neutral'
timeline.loc[timeline['spy_log_return'] > q('spy_log_return', 0.70), 'direction_tag'] = 'rally'
timeline.loc[timeline['spy_log_return'] < q('spy_log_return', 0.30), 'direction_tag'] = 'selloff'
timeline['regime_descriptor'] = timeline['stress_ladder'] + '_' + timeline['direction_tag']
timeline.to_csv(out_table_dir / 'phase2_stress_ladder_timeline_notebook.csv', index=False)
timeline[['date', target, 'stress_score', 'stress_ladder', 'direction_tag', 'regime_descriptor'] + flag_cols].head()

In [ ]:
ladder_counts = timeline.groupby(['stress_ladder', 'direction_tag']).size().reset_index(name='n')
ladder_counts.to_csv(out_table_dir / 'phase2_stress_ladder_counts_notebook.csv', index=False)
ladder_counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(timeline['date'], timeline[target], label='future_rv_20d', linewidth=1.2)
ax2 = ax.twinx()
ax2.step(timeline['date'], timeline['stress_score'], where='mid', label='stress score', alpha=0.8)
ax.axhline(thresholds['extreme_vol_q90'], linestyle='--', label='future vol q90')
ax.set_title('Stress score overlay versus future realized volatility')
ax.set_ylabel(target)
ax2.set_ylabel('stress score')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_stress_score_overlay.png', dpi=180)
plt.show()

## 6. Evaluate warning thresholds

This evaluates whether the ladder is useful for future high-volatility / extreme-volatility events. It does not replace volatility forecasting metrics.

In [ ]:
eval_rows = []
for thr in [1, 2, 3, 4]:
    signal = timeline['stress_score'] >= thr
    for event in events:
        eval_rows.append({'signal': f'stress_score >= {thr}', 'event': event, 'threshold': thr, **flag_metrics(signal, timeline[event])})
ladder_eval = pd.DataFrame(eval_rows).sort_values(['event', 'f1'], ascending=[True, False])
ladder_eval.to_csv(out_table_dir / 'phase2_stress_ladder_evaluation_notebook.csv', index=False)
ladder_eval

In [ ]:
plot_eval = ladder_eval[ladder_eval['event'] == 'future_extreme_vol_event_q90'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot_eval['threshold'], plot_eval['precision'], marker='o', label='precision')
ax.plot(plot_eval['threshold'], plot_eval['recall'], marker='o', label='recall')
ax.plot(plot_eval['threshold'], plot_eval['f1'], marker='o', label='F1')
ax.set_title('Stress-score threshold tradeoff for future q90 volatility events')
ax.set_xlabel('stress score threshold')
ax.set_ylabel('metric')
ax.set_xticks(plot_eval['threshold'])
ax.legend()
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_stress_ladder_threshold_tradeoff.png', dpi=180)
plt.show()

## 7. Interpretation checkpoint

Use this section to decide whether the four-level ladder is defensible.

Required checks:

- Do known/high-volatility episodes activate multiple stress indicators?
- Does a higher stress score improve precision while lower levels preserve recall?
- Are `crisis_like` periods mostly selloff stress, or do some occur during volatile rallies?
- Does the ladder create a useful industry-facing danger-zone sequence without pretending to discover true latent regimes?

Recommended language if the diagnostics look reasonable:

> We define an operational stress-warning ladder rather than a full market-regime taxonomy. The ladder combines future-volatility risk with observable market-stress confirmations such as VIX, drawdown, selloff pressure, and volume stress. A separate direction tag distinguishes selloff stress from volatile rallies.